# 00 Environment setup
## Learning objective
Verify Python, uv and the existing package without reading private configuration.
## Technical level
L200. Expected duration: 10 minutes.
## Prerequisites and setup
Python 3.12, Git, uv, VS Code Python and Jupyter extensions. From the repository root in Bash or PowerShell run `uv sync --frozen --all-extras --dev`. Select `.venv` as the notebook Python kernel. Installation needs package access once; local execution does not. Read the [notebook guide](README.md) for lab order, consent rules and the production/WAF disclaimer.
## Cost and permissions
Local cells have no Azure cost or license requirement. Live paths require your own authorized tenant, existing project/model/Search, quota and data-plane permissions. Agent 365 licensing is optional, not part of core Foundry.

## Read the environment boundary before running anything

An environment check answers a narrower question than a service health test:
can this kernel import the installed workshop package and construct safe local
settings? A successful answer says nothing about an Azure account, model quota,
a Search index, or a native Agent 365 inventory. Start here even if another
terminal already runs the application. Jupyter kernels can outlive terminal
activations, and a notebook opened from a different directory can accidentally
select another interpreter. The path assertion is therefore a usability guard
as well as a protection against writing evidence outside the intended project.

The frozen environment should already be prepared before the workshop. Do not
install packages inside a notebook or resolve an import error by switching to
system Python. The dependency list includes Azure libraries because later labs
have opt-in extensions; importing a library is not permission to authenticate.
The local execution contract is stronger: notebooks 00, 02, 03, 04 and 06 run
with sockets and credential acquisition denied. Their committed live switches
must remain False. A green local run is evidence of that boundary, not evidence
that a cloud service is reachable.

Keep three configurations conceptually separate. `Settings.local()` is the
explicit offline baseline. The EQUIPMENT compatibility profile uses its
existing agent/index settings and a pinned prompt agent without remote tools.
The PRESENTATION profile uses `presentation_agent_name` (default
`contoso-policy-assistant`), `presentation_search_index` (default
`fsa-presentation-synthetic`), and a separately pinned
`presentation_agent_version` returned by authorized setup. Never repurpose the
equipment settings merely to make the new path work. Configuring a version is
not publication, consent, or an enterprise approval grant.

### Step 1 — Validate the kernel and offline configuration

**Implementation and input:** the cell imports
`foundry_security_accelerator.config.Settings`, plus standard-library path,
interpreter, executable and installed-distribution checks. It takes no user
text or credentials and explicitly calls `Settings.local()`, not a dotenv
dump. **Output:** a small mapping containing package version, mode and
`azure: NOT_CHECKED`; no resource identifiers.

**Why the assertions exist:** the project marker proves the working-directory
assumption; the Python/uv checks prove prerequisites are present; the mode and
exporter assertions prove the constructed local settings cannot intentionally
request remote export. None proves token validity, SDK caller identity, cloud
support, or ingestion. If a check fails, choose the correct prepared kernel and
restart it. Do not weaken assertions to make a demonstration look successful.
The variables created here are for this local cell only; the optional cell
below initializes its own inputs and can be evaluated independently.

In [ ]:
import shutil
import sys
from importlib.metadata import version
from pathlib import Path

from foundry_security_accelerator.config import Settings

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository"
assert sys.version_info >= (3, 12), "Select the Python 3.12+ kernel"
assert shutil.which("uv"), "Install uv using its official instructions"
settings = Settings.local()
assert settings.mode == "local" and not settings.export_monitor and not settings.otlp_endpoint
print(
    {
        "package": version("foundry-security-accelerator"),
        "mode": settings.mode,
        "azure": "NOT_CHECKED",
    }
)

## Azure selection (manual and optional)
Authenticate privately with `az login`; explicitly select your own approved cloud/subscription. Do not paste login or account output into notebooks. Configure ignored `.env` using the variable names in the [live lab](../docs/participant-lab/azure-live-lab.md).
In an authorized terminal, use `uv run python -m fsa --mode azure doctor`, then `uv run python -m fsa azure whoami` and `uv run python -m fsa azure preflight`. Doctor checks configuration, not service connectivity; whoami checks redacted CLI account selection, not effective SDK identity. Preflight normally returns nonzero because capability checks remain verification-required. No resource creation is included.
## Expected results and validation
Local checks above must pass. Azure state remains NOT_CHECKED until you perform independent read-only checks. Never print settings, environment dictionaries or credentials.
## Security explanation
Explicit local settings exclude private dotenv and remote exporters. Synthetic identity is not Entra/OBO authorization.
## Troubleshooting
Wrong import/kernel: select `.venv` and rerun frozen sync. Missing Azure permissions: use local fallback; do not broaden tenant roles.
## Cleanup
No cloud resources were created. Clear outputs before sharing. See [cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[Settings](../src/foundry_security_accelerator/config.py), [preflight](../src/foundry_security_accelerator/azure_ops.py).

## Optional independent step — Azure configuration only, no service call

**Objective:** validate private Settings construction without showing any
configuration or identifiers. **Exact implementation:** only
`foundry_security_accelerator.config.Settings`; `Path` locates the private
dotenv. Input is that local file, output is a fixed accepted/not-executed
label. There is no credential acquisition, endpoint request, repr, dictionary
conversion, environment dump or service-health claim.

The committed False switch skips everything sensitive, including input().
When deliberately enabled, the confirmation authorizes local configuration
loading only. A validation error is sanitized rather than printing Pydantic's
input values. There is no assertion of connectivity: the cell's success proves
only that Settings accepted the supplied fields. Missing presentation version
may still require independently authorized setup/pinning in 01.

In [ ]:
RUN_AZURE_LIVE = False
if RUN_AZURE_LIVE:
    from pathlib import Path

    from foundry_security_accelerator.config import Settings

    if input("Load Azure configuration locally only? Type VALIDATE CONFIG ONLY: ") != (
        "VALIDATE CONFIG ONLY"
    ):
        raise PermissionError("Configuration loading not authorized")
    config_root = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
    try:
        config_only_settings = Settings(mode="azure", _env_file=config_root / ".env")
    except (ValueError, OSError):
        raise RuntimeError(
            "Configuration rejected; review privately without sharing values"
        ) from None
    print("Azure Settings accepted; connectivity, credentials and identifiers NOT_CHECKED")
else:
    print("Azure configuration validation NOT_EXECUTED; default switch is OFF")

## Interpretation and a participant decision record

| Observation | Supported interpretation | Unsupported inference |
|---|---|---|
| Local assertions pass | Prepared kernel/package can run offline checks | Azure credentials work |
| Azure configuration accepted | Settings validation accepted supplied fields | Account selection or network verified |
| Missing pinned presentation version | Setup/pin needs separate review | Use latest or overwrite equipment |
| Live switch remains False | Optional step was NOT_EXECUTED | Cloud test passed |

Record the question you wanted to answer before choosing a service check.
For example, “is my kernel correct?” needs no token; “can this principal query
this approved index?” needs a separately authorized read. “May this tenant
publish an agent?” is a governance question for the owner and administrator,
not a prerequisite for local learning. This avoids escalating permissions to
solve a problem that was really an interpreter or configuration mismatch.

## Questions and safe exercises

1. Why can a terminal import succeed while a notebook import fails? Identify
   the selected interpreter without printing its environment or dotenv.
2. Explain why configuration validation is neither authentication nor
   connectivity. List one independent observation required for each.
3. Draw separate boxes for equipment settings and presentation settings.
   Which values must never be repointed to reuse a shared resource?
4. In a private scratch discussion, classify a missing deployment, expired
   credential and denied workspace query as different problems. Do not create
   deliberate cloud failures merely to answer the exercise.

## Least privilege, cost, cleanup and next steps

This configuration-only step needs no Azure role and spends no model tokens.
Keep that least-privilege result: do not call doctor/preflight/service discovery
from the added cell. Existing private Azure configuration may include
sensitive addresses even when it contains no secret, so do not print Settings,
`repr`, `dict`, `model_dump`, exception details, environment variables, or IDs.
On validation error, the cell raises a generic message; review the cause only
in your approved private environment.

No cloud resource is created here. Restart the kernel to discard loaded
private settings, reset live flags to False and clear every output before
saving or sharing. Continue to 02–04 for offline controls or 01 for specifically
authorized live work. Dependency installation, Azure login, resource creation
and native consent are outside this cell's authority.

## Current platform references and claim boundaries

References checked on 2026-11-12; these are reading links, not remote code to execute.
[Foundry function calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)
documents the model-request/client-execution loop. The PRESENTATION profile attaches
four function definitions directly; no Toolbox is required. The model requests
`search_policies`, `get_policy_section`, `get_ticket_status`, or
`create_records_request`; application policy checks each proposed execution first.
Function selection is not authorization. Only the approved synthetic Azure Search
corpus is eligible, and a records request enters a local persistent approval queue.
Even in Azure mode, supplied action receipts are **DRY_RUN**, and the identity label
is **SYNTHETIC_ROLE_LABELS_NOT_ENTRA_OBO**, not authenticated enterprise approval.
The existing `presentation_runtime.list_approvals(...)` and
`presentation_runtime.approve_request(...)` operate on the shared live/local
persistent queue. Inspect their documented inputs privately before use; these
notebooks never automatically approve requests. Synthetic reviewer labels are
not authenticated humans. Function schemas may be visible in the Foundry portal,
but the playground does not execute this application's local function handlers.
Use the authorized application client loop, not a portal chat, to test execution.

Native scope is **read-only evidence checks and detailed portal/SDK guidance**.
There is no authorization here for publishing, native configuration, tenant
consent, RBAC, licensing or governance mutations. Telemetry export configuration
requires separate reviewed approval. Added live invocation cells require an
independent default-off authorization for already-configured Monitor/OTLP egress;
invocation consent alone does not permit export. No cell configures an exporter.
The six-scene initial verification budget is at most 18 model calls across the
entire initial verification, not 18 per notebook. Choose the cells you need and
subtract any prior calls; do not run all optional examples within that budget.

[Agent 365 SDK overview](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-sdk?tabs=python)
states that supported platforms use native integration without adding the SDK.
The SDK extends an existing runtime with identity, governed Work IQ tooling, and
notifications; it does not build, host, or orchestrate agents.
[Microsoft OpenTelemetry Distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry)
is the separate current observability integration. Do **not** add the deprecated
Agent 365 observability SDK. These references describe platform choices, not
packages installed or tenant capabilities validated by this notebook.

Official source repositories for further study:
[Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
(agent orchestration), [Azure AI Projects SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects)
(Foundry client contracts), [Agent 365 Python SDK](https://github.com/microsoft/Agent365-python)
(selected developer integrations), and
[Microsoft OpenTelemetry Python Distro](https://github.com/microsoft/opentelemetry-distro-python)
(current observability). Follow this repository's pinned environment rather than
installing arbitrary versions from sample pages. No upstream code is copied here.